In [45]:
!pip install pandas scikit-learn nltk
!pip install imbalanced-learn

In [46]:
import pandas as pd
import re
import string
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

In [47]:
file_path = "data/raw/All_emotions.csv"

df = pd.read_csv(file_path)

print("shape:", df.shape)
df.head()

shape: (7261, 3)


,speaker,utterance,emotion
0,patient,I can feel the cold water on my teeth.,neutral
1,dentist,It's important to keep the area clean.,neutral
2,patient,I try to brush twice a day like you said.,neutral
3,dentist,You might feel a bit of pressure.,neutral
4,patient,Should I open wider?,neutral


In [48]:
print(df.columns.tolist())

['speaker', 'utterance', 'emotion']


In [49]:
print(df["speaker"].unique())
print(df["speaker"].value_counts())

['patient' 'dentist']
speaker
patient    3636
dentist    3625
Name: count, dtype: int64


In [50]:
df["speaker"] = df["speaker"].astype(str).str.lower().str.strip()

df = df[df["speaker"] == "patient"]

print("After filtering only patient data:", df.shape)
df.head()

After filtering only patient data: (3636, 3)


,speaker,utterance,emotion
0,patient,I can feel the cold water on my teeth.,neutral
2,patient,I try to brush twice a day like you said.,neutral
4,patient,Should I open wider?,neutral
6,patient,I am not paying for a mistake you made.,angry
8,patient,I can feel the cold water on my teeth.,neutral


In [51]:
df = df[["utterance", "emotion"]].copy()
df.head()

,utterance,emotion
0,I can feel the cold water on my teeth.,neutral
2,I try to brush twice a day like you said.,neutral
4,Should I open wider?,neutral
6,I am not paying for a mistake you made.,angry
8,I can feel the cold water on my teeth.,neutral


In [52]:
df.dropna(subset=["utterance", "emotion"], inplace=True)

df["utterance"] = df["utterance"].astype(str)
df["emotion"] = df["emotion"].astype(str)

df = df[df["utterance"].str.strip() != ""]
df = df[df["emotion"].str.strip() != ""]

print("Shape after removing missing and empty values:", df.shape)

Shape after removing missing and empty values: (3636, 2)


In [53]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"\d+", "", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_text"] = df["utterance"].apply(clean_text)

df = df[df["clean_text"].str.strip() != ""]

df[["utterance", "clean_text", "emotion"]].head()

,utterance,clean_text,emotion
0,I can feel the cold water on my teeth.,i can feel the cold water on my teeth,neutral
2,I try to brush twice a day like you said.,i try to brush twice a day like you said,neutral
4,Should I open wider?,should i open wider,neutral
6,I am not paying for a mistake you made.,i am not paying for a mistake you made,angry
8,I can feel the cold water on my teeth.,i can feel the cold water on my teeth,neutral


In [54]:
def tokenize(text):
    return text.split()

df["tokens"] = df["clean_text"].apply(tokenize)

df[["clean_text", "tokens"]].head()

,clean_text,tokens
0,i can feel the cold water on my teeth,"[i, can, feel, the, cold, water, on, my, teeth]"
2,i try to brush twice a day like you said,"[i, try, to, brush, twice, a, day, like, you, ..."
4,should i open wider,"[should, i, open, wider]"
6,i am not paying for a mistake you made,"[i, am, not, paying, for, a, mistake, you, made]"
8,i can feel the cold water on my teeth,"[i, can, feel, the, cold, water, on, my, teeth]"


In [55]:
X_text = df["clean_text"]
y = df["emotion"]

print("Number of samples:", len(X_text))
print("\nLabel distribution:")
print(y.value_counts())

Number of samples: 3636

Label distribution:
emotion
neutral    2856
disgust     247
fear        195
angry       156
sad          98
happy        84
Name: count, dtype: int64


In [56]:
vectorizer = TfidfVectorizer(
    max_features=3000,
    ngram_range=(1, 2),
    min_df=1
)

X_tfidf = vectorizer.fit_transform(X_text)

print("TF-IDF shape:", X_tfidf.shape)
print("\nตัวอย่าง feature 20 ตัวแรก:")
print(vectorizer.get_feature_names_out()[:20])

TF-IDF shape: (3636, 3000)

ตัวอย่าง feature 20 ตัวแรก:
['able' 'able to' 'about' 'about how' 'about it' 'about looking'
 'about losing' 'about my' 'about needing' 'about that' 'about the'
 'about this' 'about to' 'about two' 'about what' 'about year'
 'absolutely' 'absolutely revolting' 'accident' 'ache']


In [57]:
tfidf_sample = pd.DataFrame(
    X_tfidf[:5].toarray(),
    columns=vectorizer.get_feature_names_out()
)

tfidf_sample.head()

,able,able to,about,about how,about it,about looking,about losing,about my,about needing,about that,...,you were,you will,you would,youll,youll see,your,your receptionist,your time,youre,youre right
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [58]:
df.to_csv("data/processed/cleaned_emotions_patient.csv", index=False, encoding="utf-8-sig")

tfidf_df = pd.DataFrame(
    X_tfidf.toarray(),
    columns=vectorizer.get_feature_names_out()
)

tfidf_df["emotion"] = y.values

tfidf_df.to_csv("data/processed/tfidf_features_patient.csv", index=False, encoding="utf-8-sig")

print("Files saved successfully:")
print("- cleaned_emotions_patient.csv")
print("- tfidf_features_patient.csv")

Files saved successfully:
- cleaned_emotions_patient.csv
- tfidf_features_patient.csv


### Train / Test Split

In [59]:
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
print("Before SMOTE")
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("\nTrain label distribution:")
print(y_train.value_counts())

print("\nTest label distribution:")
print(y_test.value_counts())

Before SMOTE
X_train shape: (2908, 3000)
X_test shape: (728, 3000)

Train label distribution:
emotion
neutral    2284
disgust     198
fear        156
angry       125
sad          78
happy        67
Name: count, dtype: int64

Test label distribution:
emotion
neutral    572
disgust     49
fear        39
angry       31
sad         20
happy       17
Name: count, dtype: int64


### Convert sparse matrix to dense

In [60]:
X_train_patient = X_train.toarray()
X_test_patient = X_test.toarray()

### Apply SMOTE on training set only

In [61]:
smote = SMOTE(random_state=42)

X_train_patient_sm, y_train_patient_sm = smote.fit_resample(
    X_train_patient, y_train
)

print("\nAfter SMOTE")
print("X_train_patient_sm shape:", X_train_patient_sm.shape)

print("\nTrain label distribution after SMOTE:")
print(pd.Series(y_train_patient_sm).value_counts())



After SMOTE
X_train_patient_sm shape: (13704, 3000)

Train label distribution after SMOTE:
emotion
neutral    2284
disgust    2284
sad        2284
happy      2284
fear       2284
angry      2284
Name: count, dtype: int64


###  Final ready data

In [62]:
X_train_final = X_train_patient_sm
y_train_final = y_train_patient_sm
X_test_final = X_test_patient
y_test_final = y_test

print("\nFinal data ready for model")
print("X_train_final:", X_train_final.shape)
print("y_train_final:", y_train_final.shape)
print("X_test_final:", X_test_final.shape)
print("y_test_final:", y_test_final.shape)



Final data ready for model
X_train_final: (13704, 3000)
y_train_final: (13704,)
X_test_final: (728, 3000)
y_test_final: (728,)


### Save train/test files

In [63]:
feature_names = vectorizer.get_feature_names_out()

train_smote_df = pd.DataFrame(X_train_final, columns=feature_names)
train_smote_df["emotion"] = y_train_final.values

test_df = pd.DataFrame(X_test_final, columns=feature_names)
test_df["emotion"] = y_test_final.values

train_smote_df.to_csv("data/processed/patient_train_tfidf_smote.csv", index=False, encoding="utf-8-sig")
test_df.to_csv("data/processed/patient_test_tfidf.csv", index=False, encoding="utf-8-sig")

print("\nFiles saved successfully:")
print("- patient_train_tfidf_smote.csv")
print("- patient_test_tfidf.csv")


Files saved successfully:
- patient_train_tfidf_smote.csv
- patient_test_tfidf.csv
